# Notebook 3 — Geometric Analysis

Near-complete reuse of the colours section from
`Reference_Papers/Representation-Manifolds/3-reproduce_figures.ipynb`.
SNOMED CT ontological distances replace hue distances.

**Input files** (from notebook 2)
- `data/embeddings_normalised.csv`
- `data/concepts_filtered.csv`
- `data/ontological_distances_filtered.csv`

## 3a. Load

In [ ]:
from utils import knn_graph, largest_connected_component, interactive_3d_plot, distance_plot
from IPython.display import IFrame

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import dijkstra

In [ ]:
X        = pd.read_csv("data/embeddings_normalised.csv", header=None).values
concepts = pd.read_csv("data/concepts_filtered.csv")
D_snomed = pd.read_csv("data/ontological_distances_filtered.csv", header=None).values

print(f"X shape:        {X.shape}")
print(f"concepts shape: {concepts.shape}")
print(f"D_snomed shape: {D_snomed.shape}")

## 3b. PCA visualisation

In [ ]:
pca = PCA(n_components=3)
Xp  = pca.fit_transform(X)

print(f"Explained variance ratio (3 PCs): {pca.explained_variance_ratio_.sum():.4f}")

fig = interactive_3d_plot(
    Xp,
    labels=concepts["preferred_term"].values,
    color_values=D_snomed[0],     # colour by ontological distance from first concept
    colormap="Viridis",
    point_size=3,
)
fig.write_html("data/interactive_3d.html", include_plotlyjs=True)
IFrame("data/interactive_3d.html", width="100%", height=600)

## 3c. k-NN graph and geodesic distances

In [ ]:
k = 5
A = knn_graph(X, k)
A.data[A.data < 1e-3] = 1e-3
lcc_mask = largest_connected_component(A, verbose=True)

In [ ]:
X_lcc        = X[lcc_mask]
D_snomed_lcc = D_snomed[np.ix_(lcc_mask, lcc_mask)]
concepts_lcc = concepts[lcc_mask].reset_index(drop=True)

A_lcc = knn_graph(X_lcc, k)
DXm   = dijkstra(A_lcc, return_predecessors=False)
DXc   = 1 - squareform(pdist(X_lcc, metric="cosine"))

print(f"LCC size: {X_lcc.shape[0]}")
print(f"Geodesic distances — finite: {np.isfinite(DXm).sum()}, infinite: {(~np.isfinite(DXm)).sum()}")

In [ ]:
from utils import plot_scatter

threshold = 30

xs_f, ys_f, ls_f = [], [], []
for i in range(len(D_snomed_lcc)):
    row = D_snomed_lcc[i]
    for d in np.unique(row[row < threshold]):
        xs_f.append(d)
        ys_f.append(np.mean(DXm[i, row == d]))
        ls_f.append(hex_colors[i])

xs_f = np.array(xs_f)
ys_f = np.array(ys_f)
ls_f = np.array(ls_f)

idx = np.random.permutation(len(xs_f))
xs_f, ys_f, ls_f = xs_f[idx], ys_f[idx], ls_f[idx]

corr = np.corrcoef(xs_f, ys_f)[0, 1]

fig, ax = plot_scatter(
    xs_f, ys_f, alpha=0.1, marker_size=0.1,
    xlabel="SNOMED CT ontological distance",
    ylabel="Manifold distance",
    colors=ls_f,
    y_min=min(ys_f),
    text_box=fr"$\rho = {corr:.3f}$",
    figsize=(3.5, 3.5),
)
fig.suptitle("Manifold distance vs SNOMED CT ontological distance\n(ontological distance < 30)", y=1.02)
fig.savefig("data/plot_manifold_vs_ontological_lt30.png", dpi=300, bbox_inches="tight")

## 3d. Correlation plots

In [ ]:
anc = concepts_lcc["ancestor_count"].values
norm = plt.Normalize(anc.min(), anc.max())
hex_colors = [mcolors.to_hex(plt.cm.viridis(norm(v))) for v in anc]

# Manifold distance vs SNOMED ontological distance
fig, ax = distance_plot(
    DXm, D_snomed_lcc,
    labels=concepts_lcc["preferred_term"].values,
    colors=hex_colors,
    corr_coef="pearson",
    xlabel="SNOMED CT ontological distance",
    ylabel="Manifold distance",
)
fig.suptitle("Manifold distance vs SNOMED CT ontological distance", y=1.02)
fig.savefig("data/plot_manifold_vs_ontological.png", dpi=300, bbox_inches="tight")

In [ ]:
# Cosine similarity vs SNOMED ontological distance
fig, ax = distance_plot(
    DXc, D_snomed_lcc,
    labels=concepts_lcc["preferred_term"].values,
    colors=hex_colors,
    corr_coef="chatterjee",
    square_distances=True,
    xlabel="Squared SNOMED CT ontological distance",
    ylabel="Cosine similarity",
)
fig.suptitle("Cosine similarity vs SNOMED CT ontological distance²", y=1.02)
fig.savefig("data/plot_cosine_vs_ontological.png", dpi=300, bbox_inches="tight")